# 🎯 [DAT06_M06] 군수 결산 보고서 생성 및 승인/반려 관리 대화형 웹 대시보드 구현
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m06_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M06 |
| **미션 주제** | 군수 결산 보고서 생성 및 승인/반려 관리 대화형 웹 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 군수 결산 보고서 생성 및 승인/반려 관리 대화형 웹 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

군수 결산 보고서 생성 및 승인/반려 대화형 웹 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **결산 보고서 생성과 승인·반려 상태를 관리하는 대시보드를 구현하고, 예외 상황에서도 중단되지 않는 입력 검증기와 에러 핸들러를 갖춘다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **부대별 보고서를 생성해 작성-제출-승인-반려 상태 전이를 관리하고 현황판과 그래프로 보여 준다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/dat06_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""부대별 결산 보고서를 생성하고 제출·승인·반려 상태를 관리하는 대시보드(상태 현황판·그래프)를 구현하며, 입력 검증기와 에러 핸들러로 예외 10종을 크래시 없이 로그에 기록하고 사용자 조치 안내를 출력한다."""
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

LEDGER = pd.DataFrame(make_ledger(2))
LOG, REPORTS = [], {}


class AppError(Exception):
    def __init__(self, code, msg, guide):
        super().__init__(msg); self.code, self.msg, self.guide = code, msg, guide


def validate(unit, month, rows):
    # [TODO : 학생 실습 6] 군수 결산 보고서 생성 및 승인/반려 관리 대화형 웹 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def generate(unit, month, data=None):
    # [TODO : 학생 실습 6] 군수 결산 보고서 생성 및 승인/반려 관리 대화형 웹 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
FLOW = {"제출": ("작성", "제출"), "승인": ("제출", "승인"), "반려": ("제출", "반려"), "재작성": ("반려", "작성")}


def act(rid, action, who, reason=""):
    if rid not in REPORTS:
        raise AppError("E08", f"보고서 {rid} 없음", "보고서 번호를 목록에서 확인")
    if action not in FLOW:
        raise AppError("E09", f"알 수 없는 처리 '{action}'", "제출/승인/반려/재작성 중 선택")
    need, new = FLOW[action]
    if REPORTS[rid]["status"] != need:
        raise AppError("E10", f"현재 상태 '{REPORTS[rid]['status']}'에서 '{action}' 불가", f"'{need}' 상태에서만 가능")
    if action == "반려" and not reason.strip():
        raise AppError("E11", "반려 사유 없음", "반려 사유를 한 줄 이상 입력")
    REPORTS[rid]["status"] = new
    REPORTS[rid]["hist"].append((new, who + (f": {reason}" if reason else "")))


def safe(fn, *a, **k):
    """에러 핸들러: 모든 예외를 잡아 로그에 남기고 사용자에게 알기 쉬운 메시지와 조치를 돌려준다(크래시 없음)."""
    try:
        r = fn(*a, **k)
        LOG.append({"level": "INFO", "code": "OK", "msg": fn.__name__}); return r
    except AppError as e:
        LOG.append({"level": "ERROR", "code": e.code, "msg": e.msg, "guide": e.guide}); return None
    except Exception as e:                    # 예상 밖 오류도 기록
        LOG.append({"level": "ERROR", "code": "E99", "msg": repr(e)[:50], "guide": "운영 담당자에게 로그 전달"}); return None


# 1) 정상 흐름: 12개 부대 보고서 생성 -> 제출 -> 승인 8 / 반려 2 / 대기 2
rids = [safe(generate, u, "2026-08") for u in UNITS[:12]]
for i, rid in enumerate(rids):
    safe(act, rid, "제출", f"담당자{i % 3 + 1}")
    if i < 8:
        safe(act, rid, "승인", "보급관")
    elif i < 10:
        safe(act, rid, "반려", "보급관", "기말 재고 산식 재확인 요망")
board = pd.DataFrame([{"보고서": k, **{x: v[x] for x in ("unit", "status", "issue", "close")}} for k, v in REPORTS.items()])
print("[결산 보고서 현황판] [교육용가상자료]")
print(board.rename(columns={"unit": "부대", "status": "상태", "issue": "불출합", "close": "기말합"}).to_string(index=False))
print("상태별:", board["status"].value_counts().to_dict())
safe(act, rids[8], "재작성", "담당자")
print("반려 건 재작성 후 상태:", REPORTS[rids[8]]["status"], "/ 이력:", REPORTS[rids[8]]["hist"][-3:])

# 2) 예외 주입 10종
LOG.clear()
bad = LEDGER[(LEDGER["unit"] == UNITS[20]) & (LEDGER["month"] == "2026-08")].copy()
neg = bad.copy(); neg.iloc[0, neg.columns.get_loc("issue")] = -5
txt = bad.copy(); txt["issue"] = txt["issue"].astype(str) + "개"
nocol = bad.drop(columns=["ret"])
cases = [("부대명 오류", generate, ("가상99대대", "2026-08")), ("월 오류", generate, (UNITS[20], "2027-01")),
         ("빈 자료", generate, (UNITS[20], "2026-08", bad.iloc[0:0])), ("열 누락", generate, (UNITS[20], "2026-08", nocol)),
         ("숫자 아님", generate, (UNITS[20], "2026-08", txt)), ("음수", generate, (UNITS[20], "2026-08", neg)),
         ("중복 생성", generate, (UNITS[0], "2026-08")), ("없는 보고서", act, ("RPT-00-00", "승인", "보급관")),
         ("상태 위반(미제출 승인)", act, (rids[8], "승인", "보급관")), ("반려 사유 누락", act, (rids[11], "반려", "보급관", " "))]
crash = 0
LOG.clear()
for name, fn, args in cases:
    try:
        safe(fn, *args)
    except BaseException:
        crash += 1
errs = [l for l in LOG if l["level"] == "ERROR"]
print("\n[예외 주입 결과 - 사용자 안내 메시지]")
for (name, _, _), l in zip(cases, errs):
    print(f" {l['code']} {name:<14} {l['msg']:<28} -> 조치: {l['guide']}")
rate = len(errs) / len(cases)
print(f"예외 {len(cases)}종 중 로그 기록 {len(errs)}건(기록률 {rate:.0%}), 크래시 {crash}건")
print("로그 예시:", json.dumps(errs[0], ensure_ascii=False))
print(f"목표(크래시 0건, 에러 로그 기록률 100%): {'달성' if crash == 0 and rate == 1 else '미달'}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cnt = board["status"].map({"승인": "approved", "반려": "rejected", "제출": "pending"}).fillna("draft").value_counts()
ax[0].bar(cnt.index, cnt.values, color="tab:blue"); ax[0].set_title("Report status")
ax[1].barh(board["unit"].str[2:4], board["issue"], color="tab:gray"); ax[1].set_title("Issued quantity by unit (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[결산 보고서 현황판] [교육용가상자료]
      보고서     부대 상태  불출합   기말합
RPT-01-08 가상01대대 승인 1147  8915
RPT-02-08 가상02대대 승인  916  9139
RPT-03-08 가상03대대 승인  778  7428
RPT-04-08 가상04대대 승인  882  8357
RPT-05-08 가상05대대 승인  920  7718
RPT-06-08 가상06대대 승인  887 10211
RPT-07-08 가상07대대 승인  967  8567
RPT-08-08 가상08대대 승인  877  7738
RPT-09-08 가상09대대 반려 1056  8987
RPT-10-08 가상10대대 반려 1015  9029
RPT-11-08 가상11대대 제출 1130  6845
RPT-12-08 가상12대대 제출 1266  9330
상태별: {'승인': 8, '반려': 2, '제출': 2}
반려 건 재작성 후 상태: 작성 / 이력: [('제출', '담당자3'), ('반려', '보급관: 기말 재고 산식 재확인 요망'), ('작성', '담당자')]

[예외 주입 결과 - 사용자 안내 메시지]
 E01 부대명 오류         존재하지 않는 부대 '가상99대대'          -> 조치: 부대 목록(가상01~24대대)에서 선택
 E02 월 오류           결산월 '2027-01' 자료 없음          -> 조치: 2026-07 또는 2026-08 중 선택
 E03 빈 자료           해당 부대 자료가 비어 있음              -> 조치: 제출 부대에 원장 재제출 요청
 E04 열 누락           필수 열 'ret' 누락                -> 조치: 표준 양식을 다시 내려받아 작성
 E05 숫자 아님          'issue' 열에 숫자가 아닌 값          -> 조치: 숫자만 입력(단위 문자 제거)
 E06 음수             'issue' 열에 음수                -> 조치: 수량 0 이상으로 수정
 E07 중복 생성          이미 생성된 보고서 RPT-01-08         -> 조치: 기존 보고서를 열어 수정·재제출
 E08 없는 보고서         보고서 RPT-00-00 없음             -> 조치: 보고서 번호를 목록에서 확인
 E10 상태 위반(미제출 승인)  현재 상태 '작성'에서 '승인' 불가         -> 조치: '제출' 상태에서만 가능
 E11 반려 사유 누락       반려 사유 없음                     -> 조치: 반려 사유를 한 줄 이상 입력
예외 10종 중 로그 기록 10건(기록률 100%), 크래시 0건
로그 예시: {"level": "ERROR", "code": "E01", "msg": "존재하지 않는 부대 '가상99대대'", "guide": "부대 목록(가상01~24대대)에서 선택"}
목표(크래시 0건, 에러 로그 기록률 100%): 달성
```
</details>
